# 2. How do you evaluate a RAG chatbot?

In the malnutrition and TB lessons, evaluation meant comparing predictions with **labels**: who really developed
GAM, who really had TB. Count the hits and misses and you have sensitivity and ROC-AUC.

Our chatbot outputs a paragraph. What is the "label" for a paragraph?

| Option | How | Problem |
|---|---|---|
| **Expert reference answers** | a clinician writes the right answer to each test question; compare | slow and costly to write; one question can have many right answers; and **who checked the reference?** An unvalidated reference just moves the error |
| **Expert review of every answer** | a clinician reads each answer and grades it | the gold standard, but too slow to repeat every time we change a setting |
| **An LLM as judge, no reference** | a second, stronger LLM checks each answer against the question and the passages it came from | fast and repeatable; checked once against people before a real decision |

We use the third one every time we change something, and keep expert review for the final decision.

## Three questions a judge can answer without knowing the right answer

A RAG answer can go wrong at two places: the **search** (wrong passages) or the **model** (right passages, bad
answer). Three checks, known as the **RAG triad**, separate these, and none of them needs a reference answer:

| Check | The judge reads | It asks | A low score means |
|---|---|---|---|
| **Context relevance** | question + retrieved passages | Are these passages about the question? | **search** failed: the model never saw the right page |
| **Groundedness** (also called faithfulness) | answer + retrieved passages | Is what the answer says supported by the passages? | the **model** added something the guideline doesn't say: a hallucination |
| **Answer relevancy** | question + answer | Does the answer address what was asked? | the **model** dodged, drifted or answered a different question |

```
             context relevance                 groundedness
  question ─────────────────────── passages ─────────────────── answer
      └──────────────────────── answer relevancy ─────────────────┘
```

For clinical decision support, **groundedness** matters most: an ungrounded answer is a confident statement the
guideline doesn't support.

What none of the three can tell you: whether the **guideline itself** is right, or whether the answer left out
something important that was in a passage the search didn't find. That still needs people.

**Ragas** is an open-source library that implements these checks: each is a carefully written prompt for the judge,
plus code that turns the judge's replies into a score from 0 to 1.

In [1]:
import os
import pandas as pd

import rag
import rag_eval

pd.set_option("display.width", 200, "display.max_colwidth", None, "display.max_rows", 40)   # show full texts
assert os.environ.get("OPENROUTER_API_KEY"), "No key yet: open llm-cds/.env, paste the key after OPENROUTER_API_KEY=, save, restart the kernel"
print("assistant:", rag.Settings().model, "  judge:", rag_eval.JUDGE_MODEL)   # the judge: a different, stronger model

assistant: google/gemma-4-26b-a4b-it   judge: anthropic/claude-haiku-5.5


## The test questions

24 questions written for this workshop, in `questions.csv`: 20 that the guidelines answer and 4 they don't (to
check that the assistant refuses).

The 20 answerable questions come in two kinds of **wording** (the `wording` column):

- **guideline**: in the guideline's own terms ("cholera alert", "MUAC cut-offs", "isoniazid preventive therapy");
- **everyday**: the way a health worker or a family member might ask, without those terms ("severe watery
  diarrhoea", "upper-arm measurement", "stop them getting sick too").

Real users write the second kind. Keep the difference in mind; it matters for the search (notebook 3).

There are no reference answers: the judge doesn't need them.

In [2]:
questions = rag_eval.load_questions()
print(questions.type.value_counts().to_string(), "\n")
questions

type
answerable      20
out_of_scope     4 



,id,type,wording,question
0,chol_01,answerable,everyday,"Two grown children in the same village got severe watery diarrhoea this week and one died. Is this a warning sign of an outbreak, and how soon must I report it?"
1,chol_02,answerable,everyday,"A 7-year-old with severe watery diarrhoea has stopped throwing up. Which medicine kills the germ, how much do I give, and what if that one can't be used?"
2,chol_03,answerable,everyday,"A 3-year-old with watery diarrhoea is very dry and weak and can't drink. How much fluid do I give through the vein, and how fast?"
3,chol_04,answerable,guideline,"For a reactive oral cholera vaccination campaign with the stockpile vaccines, how far apart should the two doses be given, and from what age can people be vaccinated?"
4,nutr_01,answerable,everyday,"In our survey 12% of young children are too thin, and there are no other problems such as disease outbreaks. What feeding programmes should we run?"
5,nutr_02,answerable,guideline,"In an emergency vitamin A supplementation round, what preventive dose and how often should infants aged 6–12 months and children over 1 year receive?"
6,nutr_03,answerable,everyday,Families in a camp have nothing to eat except what is handed out to them. How many calories a day should each person's share give?
7,csd_01,answerable,guideline,Which diseases are prioritized for climate-sensitive disease surveillance in Ethiopia?
8,csd_02,answerable,guideline,"How long does the malaria parasite need to develop inside the mosquito at about 25 °C, and below what temperatures does this development fail to complete?"
9,csd_03,answerable,everyday,A patient came in with sudden high fever and a stiff neck. When do I count this as a possible case of the infection of the brain lining?


## Step 1. What does the search find?

Before any LLM: run only the search (the setup we built in notebook 1: PDF text, 200-word windows, keywords), and
look at which pages come back for each question.

In [3]:
baseline = rag.Settings()                      # docs="pdf", chunking="fixed", retriever="tfidf", k=5
retriever = rag.build(baseline)


def top_sources(question, k=5):
    hits = retriever.search(question, k)
    return "; ".join(rag.source_ids(hits))


questions.assign(sources=questions.question.map(top_sources))[["id", "question", "sources"]]

,id,question,sources
0,chol_01,"Two grown children in the same village got severe watery diarrhoea this week and one died. Is this a warning sign of an outbreak, and how soon must I report it?",moh_acute_malnutrition_2019:p20; moh_acute_malnutrition_2019:p52; moh_acute_malnutrition_2019:p42; ephi_cholera_2022:p14; moh_acute_malnutrition_2019:p42
1,chol_02,"A 7-year-old with severe watery diarrhoea has stopped throwing up. Which medicine kills the germ, how much do I give, and what if that one can't be used?",moh_acute_malnutrition_2019:p20; moh_acute_malnutrition_2019:p52; moh_acute_malnutrition_2019:p42; ephi_cholera_2022:p12; moh_acute_malnutrition_2019:p42
2,chol_03,"A 3-year-old with watery diarrhoea is very dry and weak and can't drink. How much fluid do I give through the vein, and how fast?",moh_acute_malnutrition_2019:p20; moh_acute_malnutrition_2019:p52; moh_acute_malnutrition_2019:p42; ephi_cholera_2022:p32; ephi_cholera_2022:p26
3,chol_04,"For a reactive oral cholera vaccination campaign with the stockpile vaccines, how far apart should the two doses be given, and from what age can people be vaccinated?",ephi_cholera_2022:p70; ephi_cholera_2022:p69; ephi_cholera_2022:p69; ephi_cholera_2022:p70; ephi_cebs_2021:p26
4,nutr_01,"In our survey 12% of young children are too thin, and there are no other problems such as disease outbreaks. What feeding programmes should we run?",ephi_emergency_nutrition_2022:p70; ephi_emergency_nutrition_2022:p46; ephi_emergency_nutrition_2022:p14; moh_tb_2017:p49; moh_tb_2017:p48
5,nutr_02,"In an emergency vitamin A supplementation round, what preventive dose and how often should infants aged 6–12 months and children over 1 year receive?",ephi_emergency_nutrition_2022:p52; ephi_emergency_nutrition_2022:p52; ephi_emergency_nutrition_2022:p84; ephi_emergency_nutrition_2022:p3; moh_acute_malnutrition_2019:p19
6,nutr_03,Families in a camp have nothing to eat except what is handed out to them. How many calories a day should each person's share give?,ephi_emergency_nutrition_2022:p15; ephi_emergency_nutrition_2022:p15; ephi_emergency_nutrition_2022:p63; moh_acute_malnutrition_2019:p112; moh_acute_malnutrition_2019:p38
7,csd_01,Which diseases are prioritized for climate-sensitive disease surveillance in Ethiopia?,ephi_climate_sensitive_diseases_2022:p55; ephi_climate_sensitive_diseases_2022:p7; ephi_climate_sensitive_diseases_2022:p54; ephi_climate_sensitive_diseases_2022:p3; ephi_climate_sensitive_diseases_2022:p52
8,csd_02,"How long does the malaria parasite need to develop inside the mosquito at about 25 °C, and below what temperatures does this development fail to complete?",ephi_climate_sensitive_diseases_2022:p42; ephi_climate_sensitive_diseases_2022:p57; ephi_climate_sensitive_diseases_2022:p55; ephi_cebs_2021:p24; moh_tb_2017:p165
9,csd_03,A patient came in with sudden high fever and a stiff neck. When do I count this as a possible case of the infection of the brain lining?,ephi_cebs_2021:p13; ephi_climate_sensitive_diseases_2022:p108; moh_acute_malnutrition_2019:p40; moh_acute_malnutrition_2019:p28; moh_tb_2017:p196


> **PAUSE (3 min).** Pick three questions. Do the sources look right (the cholera guideline for a cholera
> question, the SAM guideline for a malnutrition question)? Which questions got sources from the wrong guideline?
> What does the search return for the out-of-scope questions?

Looking at 24 rows by eye is fine once. For 7 setups, every time something changes, we need the judge.

## Step 2. One question, judged by hand

Answer one question and run each of the three checks on it, to see what the judge does.

In [4]:
question = questions.question.iloc[0]
result = rag.answer(question, retriever, baseline)
passages = result["contexts"]                   # the texts of the 5 passages the search returned
print(question, "\n")
print(result["answer"])

Two grown children in the same village got severe watery diarrhoea this week and one died. Is this a warning sign of an outbreak, and how soon must I report it? 

Yes, this is a warning sign of an outbreak, as it involves two or more people aged 2 years or older with acute watery diarrhea and severe dehydration, or dying from acute watery diarrhea, from the same area within 1 week of one another [4]. The case must be reported to the surveillance focal point or person in charge immediately, within 30 minutes, for further investigation [4].


In [5]:
from ragas.metrics.collections import AnswerRelevancy, ContextRelevance, ResponseGroundedness

judge = rag_eval.judge()
checks = {
    "context relevance": (ContextRelevance(llm=judge), dict(user_input=question, retrieved_contexts=passages)),
    "groundedness":      (ResponseGroundedness(llm=judge), dict(response=result["answer"], retrieved_contexts=passages)),
    "answer relevancy":  (AnswerRelevancy(llm=judge, embeddings=rag_eval.embeddings()),
                          dict(user_input=question, response=result["answer"])),
}
for name, (metric, inputs) in checks.items():
    score = await metric.ascore(**inputs)
    print(f"{name:18} {score.value:.2f}")

context relevance  1.00


groundedness       1.00


answer relevancy   0.89


What happens inside, roughly:

- **Context relevance:** the judge rates, twice, whether the passages are relevant to the question (0, 1 or 2);
  the two ratings are averaged and scaled to 0–1.
- **Groundedness:** the judge rates, twice, whether the answer is supported by the passages: fully (2), partly (1)
  or not (0); averaged and scaled to 0–1. An answer with one invented dose among correct ones gets about 0.5.
- **Answer relevancy:** the judge writes the questions that this answer *would* be an answer to; Ragas measures how
  close in meaning they are to the real question (with an embedding model). It also scores 0 if the answer is
  non-committal ("I'm not sure").

## Step 3. All 24 questions

`rag_eval.evaluate_setup` does the same for every question, 8 at a time, and records everything in **MLflow**:
one run for this setup, with its settings and prompt, and one **trace** per question showing the search, the LLM
call and the three scores. About a minute.

In [6]:
base = rag_eval.evaluate_setup(baseline, "1. baseline: PDF text, fixed, keywords")

2026/10/29 18:59:08 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/29 18:59:08 INFO mlflow.store.db.utils: Updating database tables


2026/10/29 18:59:10 INFO mlflow.tracking.fluent: Experiment with name 'guideline-assistant' does not exist. Creating a new experiment.


2026/10/29 18:59:10 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/24 [Elapsed: 00:00, Remaining: ?]

...

Evaluating:  88%|████████▊ | 21/24 [Elapsed: 00:27, Remaining: 00:03]

Evaluating:  96%|█████████▌| 23/24 [Elapsed: 00:32, Remaining: 00:01]

Evaluating: 100%|██████████| 24/24 [Elapsed: 00:33, Remaining: 00:00] [predict_fn: 24%, scorers: 76%]

Evaluating: 100%|██████████| 24/24 [Elapsed: 00:33, Remaining: 00:00] [predict_fn: 24%, scorers: 76%]


✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: 1. baseline: PDF text, fixed, keywords
  Run ID: d2f30217218b45d1974cd4b646c4c12d

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.

1. baseline: PDF text, fixed, keywords: context_relevance 0.74, groundedness 0.57, answer_relevancy 0.50, refused_in_scope 8, refused_out_of_scope 4   (MLflow run d2f30217)


The last line printed has five numbers. The three scores are averaged over the **20 questions the guidelines do
answer**:

- **context relevance:** all 20, answered or not. When the assistant wrongly refuses, it is usually because the
  search brought the wrong passages: that has to pull this average down, not drop out of it.
- **groundedness** and **answer relevancy:** a wrong refusal counts as **0**: the health worker got no answer.

And two counts of **refusals** (answers that are exactly "The guidelines provided do not cover this."):

- `refused_out_of_scope`: refusals to the 4 questions the guidelines don't cover. **Correct**: should be 4.
- `refused_in_scope`: refusals to questions the guidelines *do* answer. **Wrong**: each one is a missed answer;
  should be 0.

**Why the 4 out-of-scope questions are left out of the averages.** For "what insulin dose for a child with type 1
diabetes?" the right behaviour is the refusal. But a refusal says nothing the passages could support and doesn't
address the question, so the judge scores it 0: averaging it in would punish honesty. They are counted instead.

Every refusal, one by one:

In [7]:
refusals = base[base.refused][["id", "type", "question", "sources"]]
refusals.insert(2, "verdict", refusals.type.map({"out_of_scope": "correct: the guidelines don't cover it",
                                                 "answerable": "WRONG: the guidelines answer this"}))
print(f"{(refusals.type == 'out_of_scope').sum()} correct refusals (of 4), "
      f"{(refusals.type == 'answerable').sum()} wrong refusals (of 20)")
refusals

4 correct refusals (of 4), 8 wrong refusals (of 20)


,id,type,verdict,question,sources
1,chol_02,answerable,WRONG: the guidelines answer this,"A 7-year-old with severe watery diarrhoea has stopped throwing up. Which medicine kills the germ, how much do I give, and what if that one can't be used?",moh_acute_malnutrition_2019:p20; moh_acute_malnutrition_2019:p52; moh_acute_malnutrition_2019:p42; ephi_cholera_2022:p12; moh_acute_malnutrition_2019:p42
2,chol_03,answerable,WRONG: the guidelines answer this,"A 3-year-old with watery diarrhoea is very dry and weak and can't drink. How much fluid do I give through the vein, and how fast?",moh_acute_malnutrition_2019:p20; moh_acute_malnutrition_2019:p52; moh_acute_malnutrition_2019:p42; ephi_cholera_2022:p32; ephi_cholera_2022:p26
6,nutr_03,answerable,WRONG: the guidelines answer this,Families in a camp have nothing to eat except what is handed out to them. How many calories a day should each person's share give?,ephi_emergency_nutrition_2022:p15; ephi_emergency_nutrition_2022:p15; ephi_emergency_nutrition_2022:p63; moh_acute_malnutrition_2019:p112; moh_acute_malnutrition_2019:p38
11,cebs_02,answerable,WRONG: the guidelines answer this,"A baby of ten days stopped breastfeeding, the jaw is locked and the body goes rigid with spasms. Which disease could this be?",ephi_emergency_nutrition_2022:p58; moh_tb_2017:p58; moh_tb_2017:p59; ephi_emergency_nutrition_2022:p58; ephi_cholera_2022:p68
14,sam_01,answerable,WRONG: the guidelines answer this,"A child in the outpatient therapeutic programme still hasn't met the cure criteria. After how long is she classed as a non-responder, and when would a child instead be counted as a defaulter?",moh_acute_malnutrition_2019:p106; moh_acute_malnutrition_2019:p37; moh_acute_malnutrition_2019:p11; moh_acute_malnutrition_2019:p13; ephi_emergency_nutrition_2022:p5
15,sam_02,answerable,WRONG: the guidelines answer this,"A badly wasted toddler is eating well and has no other illness. Which germ-killing medicine is given to every such child, how much, and for how many days?",moh_acute_malnutrition_2019:p41; ephi_cebs_2021:p13; ephi_cholera_2022:p32; moh_acute_malnutrition_2019:p116; ephi_cholera_2022:p61
16,sam_03,answerable,WRONG: the guidelines answer this,"What upper-arm measurement means a pregnant or breastfeeding woman, or an old person, is moderately malnourished?",moh_tb_2017:p201; ephi_emergency_nutrition_2022:p29; moh_tb_2017:p199; moh_tb_2017:p200; moh_acute_malnutrition_2019:p13
17,tb_01,answerable,WRONG: the guidelines answer this,"Someone lives with a family member who has TB, and we want to stop them getting sick too. Which tablet do they take, how much for grown-ups and for kids, for how long, and what goes with it?",moh_tb_2017:p162; moh_tb_2017:p163; moh_tb_2017:p161; moh_tb_2017:p139; moh_tb_2017:p138
20,oos_01,out_of_scope,correct: the guidelines don't cover it,What is the first-line drug and adult dose for uncomplicated P. falciparum malaria — should I use artemether-lumefantrine?,ephi_climate_sensitive_diseases_2022:p57; ephi_climate_sensitive_diseases_2022:p57; moh_tb_2017:p43; moh_tb_2017:p44; moh_tb_2017:p182
21,oos_02,out_of_scope,correct: the guidelines don't cover it,"How many vials of antivenom should be given for a venomous snakebite, and how is it administered?",moh_tb_2017:p57; moh_tb_2017:p59; moh_tb_2017:p58; moh_tb_2017:p18; moh_tb_2017:p19


> **PAUSE (3 min).** For a wrong refusal, look at its sources: did the search find the right guideline and page?
> If not, the model was right to refuse **given what it was shown**: the failure is in the search.
> Which **wording** do the wrongly refused questions have (the `wording` column in the question table above)?
> In the run saved here, 7 of the 8 are in everyday words. We come back to this in notebook 3.

## Step 4. Read the worst answers

Averages hide the failures. Sort by the scores, then read the question, the answer and the passages, and decide
for yourself whether the judge is right.

The table has one row per question, lowest groundedness first. For a refusal (`refused` True) only context relevance
means something (were the passages any good?); its other two scores count as 0 in the averages. Look at the answered
questions first.

In [8]:
scores = base[["id", "question", "refused"] + rag_eval.METRICS].copy()
scores[rag_eval.METRICS] = scores[rag_eval.METRICS].astype(float).round(2)
scores.sort_values(["groundedness", "context_relevance"])

,id,question,refused,context_relevance,groundedness,answer_relevancy
11,cebs_02,"A baby of ten days stopped breastfeeding, the jaw is locked and the body goes rigid with spasms. Which disease could this be?",True,0.00,0.0,0.42
21,oos_02,"How many vials of antivenom should be given for a venomous snakebite, and how is it administered?",True,0.00,0.0,0.49
22,oos_03,At what ages should infants receive the rotavirus vaccine doses to help prevent diarrhoea?,True,0.00,0.0,0.51
6,nutr_03,Families in a camp have nothing to eat except what is handed out to them. How many calories a day should each person's share give?,True,0.25,0.0,0.53
20,oos_01,What is the first-line drug and adult dose for uncomplicated P. falciparum malaria — should I use artemether-lumefantrine?,True,0.25,0.0,0.52
1,chol_02,"A 7-year-old with severe watery diarrhoea has stopped throwing up. Which medicine kills the germ, how much do I give, and what if that one can't be used?",True,0.50,0.0,0.44
2,chol_03,"A 3-year-old with watery diarrhoea is very dry and weak and can't drink. How much fluid do I give through the vein, and how fast?",True,0.50,0.0,0.46
14,sam_01,"A child in the outpatient therapeutic programme still hasn't met the cure criteria. After how long is she classed as a non-responder, and when would a child instead be counted as a defaulter?",True,0.50,0.0,0.58
15,sam_02,"A badly wasted toddler is eating well and has no other illness. Which germ-killing medicine is given to every such child, how much, and for how many days?",True,0.50,0.0,0.37
16,sam_03,"What upper-arm measurement means a pregnant or breastfeeding woman, or an old person, is moderately malnourished?",True,0.50,0.0,0.44


`explain` prints one question in full: the question, the answer, its three scores, and the start of each of the 5
passages the search gave the model. The cell below uses it on the **answered question with the lowest groundedness**
(the answer least supported by its passages); its id is printed first. Which question that is can change from run
to run (the model and the judge vary a little). In the run saved in this notebook it is `nutr_01` (12% of young
children too thin, no aggravating factors): the answer says the guidelines don't say which programme to run at 12%.
They do: a table on page 30 of the emergency nutrition guideline gives targeted supplementary feeding for 10–14%
without aggravating factors. That page is **not among its five passages**, yet context relevance is 1.00: the
passages are *about* feeding programmes, just not the one that answers. A high context relevance doesn't prove the
right passage was found.

In [9]:
def explain(table, question_id, chars=600):
    """The question, the answer, the scores, and the start of each passage the answer was based on."""
    row = table.set_index("id").loc[question_id]
    print(row.question, "\n\nANSWER:", row.answer, "\n")
    print("  ".join(f"{m} {row[m]:.2f}" if pd.notna(row[m]) else f"{m} -" for m in rag_eval.METRICS), row.judge_notes)
    for n, (source, text) in enumerate(zip(row.sources.split("; "), row.passages), 1):
        print(f"\n[{n}] {source}: {text[:chars]}...")


answered = scores[~scores.refused]
worst = answered.sort_values(["groundedness", "context_relevance"]).id.iloc[0]
print(f"The answered question with the lowest groundedness: {worst}\n")
explain(base, worst)

The answered question with the lowest groundedness: nutr_01

In our survey 12% of young children are too thin, and there are no other problems such as disease outbreaks. What feeding programmes should we run? 

ANSWER: Based on the provided guidelines, a supplementary feeding programme (SFP) may be considered when the prevalence of GAM is <10% without aggravating factors [2]. Since your survey indicates 12% of young children are too thin, the guidelines do not specify which programme to run for this specific prevalence. 

context_relevance 1.00  groundedness 0.50  answer_relevancy 0.81 

[1] ephi_emergency_nutrition_2022:p70: niacin (eg pulses, nuts, dried fish) if the staple is maize or sorghum. • There is adequate access to adequate sources of riboflavin where people are dependent on a very limited diet. • There are no cases of scurvy, pellagra, beriberi or riboflavin deficiency • The prevalence of vitamin A deficiency, iron deficiency anemia and iodine deficiency disorders are not o

> **Exercise (10 min), in pairs.** Take the three answers with the lowest scores (any of the three checks).
> For each, decide:
> 1. Was it the **search** (wrong passages: low context relevance) or the **model** (good passages, but low
>    groundedness or answer relevancy)?
> 2. Do you agree with the judge? Read the passages (`explain(base, "csd_03")`, with the question's id) and check.
>
> Are there any **refusals to in-scope questions**? Look at their sources: did the search find the right page?

## Step 5. The same, in the MLflow UI

In a terminal, from `llm-cds/`, environment active:

```bash
mlflow ui            # http://127.0.0.1:5000
```

Experiment **guideline-assistant** → the run **1. baseline...** → the **Traces** (or Evaluations) tab. Click one
question: the trace shows the `search` step (the passages, with their pages and scores), the `llm` step (the exact
messages and the answer), and the three scores on the right. The run's **Parameters**
record the full setup (model, prompt version, chunking, k, judge) and a fingerprint of the passages: everything
needed to reproduce this score.

## What these three checks don't cover

> **PAUSE (5 min), discussion.**
> - The judge sees only the passages the search returned. Can it notice an answer that is grounded in those
>   passages but **incomplete**, because the important line was on a page the search missed?
> - Before this assistant reaches a health worker, what would you still want a clinician to review, and how many
>   answers?

The checks tell us whether the assistant answered **from the guideline**. Whether the guideline answer is the
**complete and right** clinical answer still needs a clinician: grade a sample of answers once, and check the
judge agrees with them, before the assistant is used for real decisions.

## Your turn

Write two questions of your own: one you expect the guidelines to answer well, and one you expect to be hard
(a table, an everyday phrasing, Amharic, or something the guidelines may not cover). Evaluate only those, then
read the scores: do they match what you expected?

In [ ]:
my_questions = [
    "",     # your first question, between the quotes
    "",     # your second question
]
assert all(q.strip() for q in my_questions), "Write your two questions between the quotes first"

mine = pd.DataFrame({"id": [f"my_{i}" for i in range(1, len(my_questions) + 1)], "type": "mine",
                     "question": my_questions})
my_scores = rag_eval.evaluate_setup(baseline, "my questions", mine)
my_scores[["question", "sources", "answer"] + rag_eval.METRICS]